# career_success_score — v7 competition feature lab
Bu sürüm `overview.md` ve `data.md` içindeki her feature grubunu ciddiye alarak ham öğrenci profilinden güçlü, leakage-kontrollü özellikler üretir.

**Problem:** `career_success_score` için 0-100 aralığında regresyon; resmi metrik MSE.

**Pipeline:** veri yükleme fallback'i → domain FE → Türkçe transformer metin özellikleri → yıl ağırlıkları / wmse → CatBoost + LightGBM + AutoGluon → ağırlıklı Ridge stack → submission.

**Metin stratejisi:**
- Hızlı semantik embedding: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` → PCA bileşenleri.
- Yarışma modu: GPU varsa `dbmdz/bert-base-turkish-cased` ve `dbmdz/electra-base-turkish-cased-discriminator` ile OOF text-only regresyon tahmini.
- Klasik yedek: keyword sentiment ve metin uzunluğu/topic bayrakları.

**Güvenli çalışma modu:**
- `train_final.csv` / `test_final.csv` varsa onları okur ve ek feature'ları üzerine ekler.
- Yoksa orijinal `train.csv` / `test_x.csv` dosyalarından kendi feature setini üretir.
- `bert_pred` varsa kullanır; yoksa transformer/embedding feature'larıyla metin sinyalini üretir.

**Leakage notları:**
1. Hazır `bert_pred` varsa train tarafında OOF olmalı; değilse CV yapay iyi görünür.
2. Target encoding fold dışı üretilir; test tarafında yalnızca train katmanından öğrenilen grup ortalamaları kullanılır.
3. Text-only transformer regresyon tahminleri de fold dışı üretilir.
4. Raw `mentor_feedback_text` modele doğrudan verilmez; türetilmiş sayısal metin özellikleri kullanılır.


In [ ]:
# Hücre 0 — ortam kontrolü; local .venv hazırsa kurulum yapmaz
import importlib.util, os, subprocess, sys, time

REQUIRED_MODULES = ['autogluon.tabular', 'catboost', 'lightgbm', 'transformers', 'sentence_transformers', 'accelerate']
missing = [m for m in REQUIRED_MODULES if importlib.util.find_spec(m) is None]

if missing:
    print('Eksik paketler:', missing)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'autogluon', 'catboost', 'lightgbm', 'transformers', 'sentence-transformers', 'accelerate',
        '--upgrade', '--no-cache-dir', '-q'
    ])
    print('Kurulum tamam. Kerneli yeniden başlatıp sonraki hücreden devam edin.')
    time.sleep(1)
    os.kill(os.getpid(), 9)
else:
    print('Ortam hazır:', sys.executable)


In [ ]:
# Hücre 1 — veri yükleme ve yarışma sözleşmesi
import warnings, time, re, gc, os
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
t0 = time.time()

RANDOM_STATE = 42
TARGET = 'career_success_score'
ID_COL = 'student_id'
LOCAL_RUN = not Path('/kaggle/working').exists()
LOCAL_TIME_LIMIT = 1800   # local AutoGluon süresi: 30 dakika
KAGGLE_TIME_LIMIT = 5400  # Kaggle/uzun koşu süresi: 90 dakika
TIME_LIMIT = LOCAL_TIME_LIMIT if LOCAL_RUN else KAGGLE_TIME_LIMIT

# Feature anahtarları: süreye göre kapatıp açılabilir.
RUN_SENTENCE_EMBEDDINGS = True
SENTENCE_EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SENTENCE_EMBEDDING_COMPONENTS = 64
RUN_TRANSFORMER_TEXT_OOF = False if LOCAL_RUN else True
TRANSFORMER_MODEL_IDS = [
    'dbmdz/bert-base-turkish-cased',
    'dbmdz/electra-base-turkish-cased-discriminator',
]
TRANSFORMER_N_SPLITS = 5
TRANSFORMER_EPOCHS = 2
TRANSFORMER_MAX_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 16
TRANSFORMER_LR = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
ALLOW_CPU_TRANSFORMER_OOF = False  # GPU yoksa text fine-tune çok yavaş; sentence embedding + keyword featureları devam eder.

CACHE_DIR = Path('/kaggle/working/text_feature_cache') if Path('/kaggle/working').exists() else Path('derived/text_feature_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'LOCAL_RUN={LOCAL_RUN} | TIME_LIMIT={TIME_LIMIT} | RUN_TRANSFORMER_TEXT_OOF={RUN_TRANSFORMER_TEXT_OOF}')


def find_csv(cands, required=True):
    roots = [Path('/kaggle/input'), Path('.'), Path('datathon-2026 (1)')]
    seen = set()
    for root in roots:
        if not root.exists() or root in seen:
            continue
        seen.add(root)
        for c in cands:
            m = sorted(root.rglob(c)) if root.is_dir() else []
            if m:
                return m[0]
    if required:
        raise FileNotFoundError(cands)
    return None


def read_csv(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    return df

train_final_path = find_csv(['train_final.csv'], required=False)
test_final_path = find_csv(['test_final.csv'], required=False)
raw_train_path = find_csv(['train.csv'], required=False)
raw_test_path = find_csv(['test_x.csv', 'test.csv'], required=False)

if train_final_path and test_final_path:
    train = read_csv(train_final_path)
    test = read_csv(test_final_path)
    print('Hazır FE dosyaları kullanılıyor:', train_final_path, test_final_path)
else:
    train = read_csv(raw_train_path)
    test = read_csv(raw_test_path)
    print('Ham yarışma dosyaları kullanılıyor:', raw_train_path, raw_test_path)

raw_train = read_csv(raw_train_path) if raw_train_path else train.copy()
raw_test = read_csv(raw_test_path) if raw_test_path else test.copy()

# Hazır FE dosyaları kullanılsa bile ham veri kolonları lazımsa geri ekle.
# Böylece mentor text / domain raw skorları kaybolmaz.
def enrich_missing_raw_columns(df, raw_df):
    if raw_df is None:
        return df
    missing_cols = [c for c in raw_df.columns if c not in df.columns and c != TARGET]
    if not missing_cols:
        return df
    df = df.copy()
    if len(df) == len(raw_df):
        for c in missing_cols:
            df[c] = raw_df[c].values
        print(f'Eksik raw kolonlar pozisyonel eklendi: {len(missing_cols)}')
        return df
    if ID_COL in df.columns and ID_COL in raw_df.columns:
        add = raw_df[[ID_COL] + missing_cols].drop_duplicates(ID_COL)
        df = df.merge(add, on=ID_COL, how='left')
        print(f'Eksik raw kolonlar ID merge ile eklendi: {len(missing_cols)}')
    return df

train = enrich_missing_raw_columns(train, raw_train)
test = enrich_missing_raw_columns(test, raw_test)

test_ids = raw_test[ID_COL].values if ID_COL in raw_test.columns else test[ID_COL].values
assert TARGET in train.columns, f'{TARGET} train içinde yok'
assert len(test_ids) == len(test), 'test satır sayısı orijinal test ile uyuşmuyor!'

for col in ['age', 'coding_score']:
    if col in raw_test.columns and col in test.columns:
        assert np.allclose(pd.to_numeric(raw_test[col], errors='coerce').fillna(-1).values,
                           pd.to_numeric(test[col], errors='coerce').fillna(-1).values), f'HIZA BOZUK: {col}'
print('Satır hizası doğrulandı')
print('train/test:', train.shape, test.shape, '| ID örnek:', test_ids[:2])

y = pd.to_numeric(train[TARGET], errors='coerce').values
print(train[TARGET].describe().round(3))
if 'application_year' in train.columns and 'application_year' in test.columns:
    print('Train yıl dağılımı:', train['application_year'].value_counts(normalize=True).sort_index().round(3).to_dict())
    print('Test yıl dağılımı :', test['application_year'].value_counts(normalize=True).sort_index().round(3).to_dict())


## 1) Feature-rich tabular + Turkish transformer text FE
Data.md'deki kolonlar ayrı bilgi aileleri: akademik sinyal, hedef rol ve teknik yetenek, gerçek proje/staj deneyimi, portfolyo/GitHub görünürlüğü, mülakat performansı, sosyal beceriler, başvuru verimliliği ve mentor metni. Bu hücre her aile için domain feature üretir; sonra Türkçe transformer tabanlı metin sinyalini ekler.


In [ ]:
# Hücre 2 — feature factory + transformer text features + OOF target encoding + drift ağırlıkları
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA

BASE_CAT = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
TECH_COLS = ['coding_score', 'problem_solving_score', 'data_structures_score', 'sql_score',
             'machine_learning_score', 'backend_score', 'frontend_score', 'cloud_score', 'devops_score']
CORE_DEV_COLS = ['coding_score', 'problem_solving_score', 'data_structures_score']
DATA_AI_COLS = ['sql_score', 'machine_learning_score', 'problem_solving_score']
WEB_DEV_COLS = ['backend_score', 'frontend_score', 'coding_score']
INFRA_COLS = ['cloud_score', 'devops_score', 'backend_score']
SOFT_COLS = ['communication_score', 'teamwork_score', 'leadership_score', 'presentation_score']
INTERVIEW_COLS = ['technical_interview_score', 'hr_interview_score']
PORTFOLIO_COLS = ['project_quality_score', 'portfolio_score', 'github_repo_count', 'github_avg_stars',
                  'open_source_contribution_count']
EXPERIENCE_COLS = ['real_client_project_count', 'internship_count', 'internship_duration_months',
                   'freelance_project_count', 'hackathon_count', 'hackathon_awards']
CAREER_PREP_COLS = ['linkedin_profile_score', 'cv_quality_score', 'certification_count', 'bootcamp_count',
                    'applications_sent', 'interviews_attended']
ACADEMIC_COLS = ['cgpa', 'english_exam_score', 'attendance_rate']
MISSING_FLAG_COLS = ['english_exam_score', 'internship_duration_months', 'github_avg_stars',
                     'open_source_contribution_count', 'hr_interview_score', 'linkedin_profile_score',
                     'portfolio_score']

ROLE_SKILL_MAP = {
    'Backend Developer': ['backend_score', 'coding_score', 'data_structures_score', 'sql_score', 'problem_solving_score'],
    'Frontend Developer': ['frontend_score', 'coding_score', 'presentation_score', 'communication_score'],
    'Software Developer': ['coding_score', 'backend_score', 'frontend_score', 'problem_solving_score', 'data_structures_score'],
    'Data Scientist': ['machine_learning_score', 'sql_score', 'problem_solving_score', 'data_structures_score'],
    'Data Analyst': ['sql_score', 'problem_solving_score', 'presentation_score', 'communication_score'],
    'AI Engineer': ['machine_learning_score', 'coding_score', 'problem_solving_score', 'data_structures_score'],
    'MLOps Engineer': ['machine_learning_score', 'devops_score', 'cloud_score', 'backend_score'],
    'DevOps Engineer': ['devops_score', 'cloud_score', 'backend_score', 'problem_solving_score'],
    'Cloud Engineer': ['cloud_score', 'devops_score', 'backend_score', 'problem_solving_score'],
    'Cybersecurity Analyst': ['devops_score', 'cloud_score', 'coding_score', 'problem_solving_score'],
    'Product Analyst': ['communication_score', 'presentation_score', 'sql_score', 'problem_solving_score'],
}

POS_KEYWORDS = ['mükemmel', 'olağanüstü', 'başarı', 'başarılar', 'güçlü', 'yüksek', 'potansiyel',
                'etkileyici', 'yaratıcı', 'analitik', 'dikkat çekici', 'avantaj', 'katkı', 'liderlik',
                'kanıtlıyor', 'sergiliyor', 'öne çıkıyor']
NEG_KEYWORDS = ['eksik', 'eksikliği', 'geliştirmesi', 'geliştirmeli', 'çalışması', 'çalışmalı',
                'ihtiyaç', 'zorlaştırıyor', 'zorluk', 'düşük', 'başlangıç', 'risk', 'gerekiyor',
                'yetersiz', 'güçlendirmesi']


def existing(cols, df):
    return [c for c in cols if c in df.columns]


def numericize(df):
    skip = set(BASE_CAT + [ID_COL, 'mentor_feedback_text'])
    for c in df.columns:
        if c not in skip and c != TARGET:
            df[c] = pd.to_numeric(df[c], errors='ignore')
    return df


def add_mean_std(df, name, cols):
    cols = existing(cols, df)
    if not cols:
        return
    block = df[cols].apply(pd.to_numeric, errors='coerce')
    df[f'{name}_mean'] = block.mean(axis=1)
    df[f'{name}_std'] = block.std(axis=1).fillna(0)
    df[f'{name}_min'] = block.min(axis=1)
    df[f'{name}_max'] = block.max(axis=1)
    df[f'{name}_range'] = df[f'{name}_max'] - df[f'{name}_min']


def safe_div(a, b):
    return a / b.replace(0, np.nan)


def add_interaction(df, a, b, name=None, scale=100.0):
    if a in df.columns and b in df.columns:
        name = name or f'{a}_x_{b}'
        df[name] = pd.to_numeric(df[a], errors='coerce') * pd.to_numeric(df[b], errors='coerce') / scale


def add_role_fit(df):
    if 'target_role' not in df.columns:
        return
    tech_cols = existing(TECH_COLS, df)
    df['role_skill_fit'] = df[tech_cols].mean(axis=1) if tech_cols else np.nan
    for role, cols in ROLE_SKILL_MAP.items():
        cols = existing(cols, df)
        if cols:
            mask = df['target_role'].astype(str).eq(role)
            df.loc[mask, 'role_skill_fit'] = df.loc[mask, cols].mean(axis=1)
            df.loc[mask, 'role_skill_min'] = df.loc[mask, cols].min(axis=1)
            df.loc[mask, 'role_skill_max'] = df.loc[mask, cols].max(axis=1)
            df.loc[mask, 'role_skill_std'] = df.loc[mask, cols].std(axis=1).fillna(0)
    if 'tech_mean' in df.columns:
        df['role_skill_gap_vs_tech'] = df['role_skill_fit'] - df['tech_mean']
    if 'project_quality_score' in df.columns:
        df['role_project_fit'] = df['role_skill_fit'] * df['project_quality_score'] / 100.0
    if 'technical_interview_score' in df.columns:
        df['role_interview_fit'] = df['role_skill_fit'] * df['technical_interview_score'] / 100.0


def add_text_features(df):
    if 'mentor_feedback_text' not in df.columns:
        return
    txt = df['mentor_feedback_text'].fillna('').astype(str)
    low = txt.str.lower()
    tokens = low.str.findall(r'[a-zçğıöşü]+')
    df['mentor_char_count'] = txt.str.len()
    df['mentor_word_count'] = tokens.str.len()
    df['mentor_unique_word_count'] = tokens.apply(lambda xs: len(set(xs)))
    df['mentor_sentence_count'] = low.str.count(r'[.!?]+').clip(lower=1)
    df['mentor_avg_word_len'] = df['mentor_char_count'] / df['mentor_word_count'].replace(0, np.nan)
    df['mentor_lexical_diversity'] = df['mentor_unique_word_count'] / df['mentor_word_count'].replace(0, np.nan)
    df['mentor_pos_kw_count'] = sum(low.str.contains(k, regex=False).astype(int) for k in POS_KEYWORDS)
    df['mentor_neg_kw_count'] = sum(low.str.contains(k, regex=False).astype(int) for k in NEG_KEYWORDS)
    df['mentor_sentiment_balance'] = df['mentor_pos_kw_count'] - df['mentor_neg_kw_count']
    df['mentor_sentiment_ratio'] = (df['mentor_pos_kw_count'] + 1) / (df['mentor_neg_kw_count'] + 1)
    df['mentor_has_but'] = low.str.contains('ancak|fakat|ama', regex=True).astype(int)
    df['mentor_has_need'] = low.str.contains('ihtiyaç|gerek|çalışmalı|geliştirm', regex=True).astype(int)
    df['mentor_has_superlative'] = low.str.contains('mükemmel|olağanüstü|çok yüksek|çok güçlü', regex=True).astype(int)
    df['mentor_has_deficit'] = low.str.contains('eksik|düşük|zorluk|yetersiz', regex=True).astype(int)

    topic_patterns = {
        'mentor_topic_backend': 'backend|veritabanı|api',
        'mentor_topic_frontend': 'frontend|arayüz',
        'mentor_topic_data': 'veri bilimi|veri analizi|analitik|sql',
        'mentor_topic_ai_ml': 'ai|yapay zeka|makine öğrenimi|ml',
        'mentor_topic_devops_cloud': 'devops|cloud|bulut',
        'mentor_topic_security': 'siber güvenlik|security',
        'mentor_topic_project': 'proje|portföy|gerçek müşteri',
        'mentor_topic_soft': 'iletişim|takım|liderlik|sunum',
        'mentor_topic_experience': 'staj|hackathon|açık kaynak|freelance',
    }
    for col, pat in topic_patterns.items():
        df[col] = low.str.contains(pat, regex=True).astype(int)

    role_patterns = {
        'Backend Developer': ['backend', 'veritabanı', 'api'],
        'Frontend Developer': ['frontend', 'arayüz'],
        'Software Developer': ['yazılım', 'software'],
        'Data Scientist': ['veri bilimi', 'makine öğrenimi', 'analitik'],
        'Data Analyst': ['veri analizi', 'analitik', 'sql'],
        'AI Engineer': ['ai', 'yapay zeka', 'makine öğrenimi'],
        'MLOps Engineer': ['mlops', 'devops', 'cloud'],
        'DevOps Engineer': ['devops', 'cloud'],
        'Cloud Engineer': ['cloud', 'bulut'],
        'Cybersecurity Analyst': ['siber güvenlik', 'security'],
        'Product Analyst': ['ürün', 'product', 'analitik'],
    }
    if 'target_role' in df.columns:
        df['mentor_mentions_target_role'] = [
            int(any(p in text for p in role_patterns.get(role, [])))
            for role, text in zip(df['target_role'].astype(str), low)
        ]


def add_feature_rich(df):
    df = df.copy()
    df = numericize(df)

    for c in BASE_CAT:
        if c in df.columns:
            df[c] = df[c].fillna('__MISSING__').astype(str)

    for c in MISSING_FLAG_COLS:
        if c in df.columns:
            df[f'{c}_missing'] = df[c].isna().astype(int)
    miss_cols = [f'{c}_missing' for c in MISSING_FLAG_COLS if f'{c}_missing' in df.columns]
    if miss_cols:
        df['total_key_missing_count'] = df[miss_cols].sum(axis=1)

    if 'university_tier' in df.columns:
        tier_num = df['university_tier'].str.extract(r'(\d+)')[0].astype(float)
        df['university_tier_num'] = tier_num
        df['university_tier_strength'] = 5 - tier_num
    if 'application_year' in df.columns:
        df['application_year_idx'] = pd.to_numeric(df['application_year'], errors='coerce') - 2019
    if {'application_year', 'graduation_year'}.issubset(df.columns):
        df['years_since_graduation'] = pd.to_numeric(df['application_year'], errors='coerce') - pd.to_numeric(df['graduation_year'], errors='coerce')
        df['is_pre_graduation'] = (df['years_since_graduation'] < 0).astype(int)
        df['is_graduation_year'] = (df['years_since_graduation'] == 0).astype(int)
    if {'age', 'graduation_year', 'application_year'}.issubset(df.columns):
        df['age_at_graduation'] = pd.to_numeric(df['age'], errors='coerce') + pd.to_numeric(df['graduation_year'], errors='coerce') - pd.to_numeric(df['application_year'], errors='coerce')
    if 'age' in df.columns:
        df['age_bucket'] = pd.cut(pd.to_numeric(df['age'], errors='coerce'), bins=[0, 21, 23, 25, 99], labels=['<=21', '22-23', '24-25', '26+']).astype(str)

    add_mean_std(df, 'academic', ACADEMIC_COLS)
    add_mean_std(df, 'tech', TECH_COLS)
    add_mean_std(df, 'core_dev', CORE_DEV_COLS)
    add_mean_std(df, 'data_ai', DATA_AI_COLS)
    add_mean_std(df, 'web_dev', WEB_DEV_COLS)
    add_mean_std(df, 'infra', INFRA_COLS)
    add_mean_std(df, 'soft', SOFT_COLS)
    add_mean_std(df, 'interview', INTERVIEW_COLS)
    add_mean_std(df, 'portfolio', PORTFOLIO_COLS)
    add_mean_std(df, 'experience', EXPERIENCE_COLS)
    add_mean_std(df, 'career_prep', CAREER_PREP_COLS)

    add_role_fit(df)

    if {'tech_mean', 'soft_mean'}.issubset(df.columns):
        df['tech_soft_gap'] = df['tech_mean'] - df['soft_mean']
        df['tech_soft_product'] = df['tech_mean'] * df['soft_mean'] / 100.0
    if {'technical_interview_score', 'hr_interview_score'}.issubset(df.columns):
        df['interview_gap_technical_hr'] = df['technical_interview_score'] - df['hr_interview_score']
    if {'project_quality_score', 'portfolio_score'}.issubset(df.columns):
        df['portfolio_project_gap'] = df['portfolio_score'] - df['project_quality_score']
    if {'cgpa', 'failed_courses_count'}.issubset(df.columns):
        df['academic_risk_score'] = (4.0 - pd.to_numeric(df['cgpa'], errors='coerce')) * (1 + pd.to_numeric(df['failed_courses_count'], errors='coerce'))
    if {'attendance_rate', 'failed_courses_count'}.issubset(df.columns):
        df['attendance_failure_pressure'] = (100 - pd.to_numeric(df['attendance_rate'], errors='coerce')) * (1 + pd.to_numeric(df['failed_courses_count'], errors='coerce'))
    if {'applications_sent', 'interviews_attended'}.issubset(df.columns):
        df['interview_conversion_rate'] = safe_div(pd.to_numeric(df['interviews_attended'], errors='coerce'), pd.to_numeric(df['applications_sent'], errors='coerce'))
        df['applications_without_interview'] = pd.to_numeric(df['applications_sent'], errors='coerce') - pd.to_numeric(df['interviews_attended'], errors='coerce')
    if {'hackathon_awards', 'hackathon_count'}.issubset(df.columns):
        df['hackathon_award_rate'] = safe_div(pd.to_numeric(df['hackathon_awards'], errors='coerce'), pd.to_numeric(df['hackathon_count'], errors='coerce'))
    if {'internship_duration_months', 'internship_count'}.issubset(df.columns):
        df['months_per_internship'] = safe_div(pd.to_numeric(df['internship_duration_months'], errors='coerce'), pd.to_numeric(df['internship_count'], errors='coerce'))
        df['has_internship'] = (pd.to_numeric(df['internship_count'], errors='coerce') > 0).astype(int)
    if {'github_avg_stars', 'github_repo_count'}.issubset(df.columns):
        df['github_total_stars_proxy'] = pd.to_numeric(df['github_avg_stars'], errors='coerce') * pd.to_numeric(df['github_repo_count'], errors='coerce')
    if {'open_source_contribution_count', 'github_repo_count'}.issubset(df.columns):
        df['oss_per_repo'] = safe_div(pd.to_numeric(df['open_source_contribution_count'], errors='coerce'), pd.to_numeric(df['github_repo_count'], errors='coerce'))
    if {'real_client_project_count', 'freelance_project_count', 'hackathon_count'}.issubset(df.columns):
        df['external_project_count'] = pd.to_numeric(df['real_client_project_count'], errors='coerce') + pd.to_numeric(df['freelance_project_count'], errors='coerce') + pd.to_numeric(df['hackathon_count'], errors='coerce')
    if {'certification_count', 'bootcamp_count'}.issubset(df.columns):
        df['formal_learning_count'] = pd.to_numeric(df['certification_count'], errors='coerce') + pd.to_numeric(df['bootcamp_count'], errors='coerce')

    for a, b, name in [
        ('project_quality_score', 'technical_interview_score', 'project_x_technical_interview'),
        ('project_quality_score', 'portfolio_score', 'project_x_portfolio'),
        ('communication_score', 'technical_interview_score', 'communication_x_technical_interview'),
        ('communication_score', 'hr_interview_score', 'communication_x_hr_interview'),
        ('tech_mean', 'project_quality_score', 'tech_x_project_quality'),
        ('tech_mean', 'technical_interview_score', 'tech_x_technical_interview'),
        ('soft_mean', 'hr_interview_score', 'soft_x_hr_interview'),
        ('portfolio_mean', 'career_prep_mean', 'portfolio_x_career_prep'),
        ('experience_mean', 'project_quality_score', 'experience_x_project_quality'),
        ('role_skill_fit', 'technical_interview_score', 'role_fit_x_technical_interview'),
    ]:
        add_interaction(df, a, b, name=name)

    if {'real_client_project_count', 'project_quality_score'}.issubset(df.columns):
        df['client_project_quality'] = np.log1p(pd.to_numeric(df['real_client_project_count'], errors='coerce')) * df['project_quality_score']

    for c in ['applications_sent', 'interviews_attended', 'github_repo_count', 'github_avg_stars',
              'open_source_contribution_count', 'real_client_project_count', 'freelance_project_count',
              'hackathon_count', 'hackathon_awards', 'certification_count', 'bootcamp_count']:
        if c in df.columns:
            df[f'log1p_{c}'] = np.log1p(pd.to_numeric(df[c], errors='coerce'))

    if {'department', 'target_role'}.issubset(df.columns):
        df['department__target_role'] = df['department'].astype(str) + '__' + df['target_role'].astype(str)
    if {'university_tier', 'target_role'}.issubset(df.columns):
        df['tier__target_role'] = df['university_tier'].astype(str) + '__' + df['target_role'].astype(str)
    if {'age_bucket', 'target_role'}.issubset(df.columns):
        df['age_bucket__target_role'] = df['age_bucket'].astype(str) + '__' + df['target_role'].astype(str)
    if {'department', 'university_tier'}.issubset(df.columns):
        df['department__tier'] = df['department'].astype(str) + '__' + df['university_tier'].astype(str)

    add_text_features(df)
    return df


def add_frequency_encoding(train_df, test_df, cols):
    all_df = pd.concat([train_df[cols], test_df[cols]], axis=0, ignore_index=True)
    for c in cols:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        freq = all_df[c].astype(str).value_counts(normalize=True)
        train_df[f'{c}_freq'] = train_df[c].astype(str).map(freq).astype(float)
        test_df[f'{c}_freq'] = test_df[c].astype(str).map(freq).astype(float)
    return train_df, test_df


def add_group_rank_features(train_df, test_df):
    score_cols = existing([
        'project_quality_score', 'technical_interview_score', 'role_skill_fit', 'tech_mean',
        'portfolio_mean', 'experience_mean', 'career_prep_mean', 'mentor_sentiment_balance',
        'github_total_stars_proxy', 'external_project_count'
    ], train_df)
    group_cols = existing(['application_year', 'target_role', 'department__target_role', 'tier__target_role'], train_df)
    if not score_cols or not group_cols:
        return train_df, test_df
    all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    n_train = len(train_df)
    for g in group_cols:
        gname = re.sub(r'[^a-zA-Z0-9]+', '_', g).strip('_')
        for c in score_cols:
            cname = re.sub(r'[^a-zA-Z0-9]+', '_', c).strip('_')
            vals = pd.to_numeric(all_df[c], errors='coerce')
            grp = all_df[g].astype(str)
            rank_col = f'{cname}_rank_in_{gname}'
            z_col = f'{cname}_z_in_{gname}'
            all_df[rank_col] = vals.groupby(grp).rank(pct=True)
            mean = vals.groupby(grp).transform('mean')
            std = vals.groupby(grp).transform('std').replace(0, np.nan)
            all_df[z_col] = (vals - mean) / std
            train_df[rank_col] = all_df.loc[:n_train-1, rank_col].values
            test_df[rank_col] = all_df.loc[n_train:, rank_col].values
            train_df[z_col] = all_df.loc[:n_train-1, z_col].values
            test_df[z_col] = all_df.loc[n_train:, z_col].values
    return train_df, test_df


def cache_slug(text):
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')[:80]


def add_sentence_embedding_features(train_df, test_df):
    if not RUN_SENTENCE_EMBEDDINGS or 'mentor_feedback_text' not in train_df.columns:
        return train_df, test_df
    cache_path = CACHE_DIR / f'sentence_embeddings_{cache_slug(SENTENCE_EMBEDDING_MODEL)}_{SENTENCE_EMBEDDING_COMPONENTS}.npz'
    if cache_path.exists():
        data = np.load(cache_path, allow_pickle=True)
        comps_train, comps_test = data['train'], data['test']
        print('Sentence embedding cache kullanıldı:', cache_path)
    else:
        try:
            from sentence_transformers import SentenceTransformer
            import torch
        except Exception as e:
            print('SentenceTransformer import edilemedi, embedding atlandı:', repr(e))
            return train_df, test_df
        texts = pd.concat([
            train_df['mentor_feedback_text'].fillna('').astype(str),
            test_df['mentor_feedback_text'].fillna('').astype(str)
        ], ignore_index=True).tolist()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        batch_size = 128 if device == 'cuda' else 32
        model = SentenceTransformer(SENTENCE_EMBEDDING_MODEL, device=device)
        emb = model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
        emb = np.asarray(emb, dtype=np.float32)
        n_comp = min(SENTENCE_EMBEDDING_COMPONENTS, emb.shape[1], len(emb) - 1)
        pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        comps = pca.fit_transform(emb).astype(np.float32)
        comps_train, comps_test = comps[:len(train_df)], comps[len(train_df):]
        np.savez_compressed(cache_path, train=comps_train, test=comps_test)
        print(f'Sentence embedding eklendi: {SENTENCE_EMBEDDING_MODEL} | dim={emb.shape[1]} | PCA={n_comp} | explained={pca.explained_variance_ratio_.sum():.3f}')
        del model, emb, comps
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
    for i in range(comps_train.shape[1]):
        col = f'st_emb_{i:02d}'
        train_df[col] = comps_train[:, i]
        test_df[col] = comps_test[:, i]
    train_df['st_emb_abs_mean'] = np.abs(comps_train).mean(axis=1)
    test_df['st_emb_abs_mean'] = np.abs(comps_test).mean(axis=1)
    train_df['st_emb_l2'] = np.sqrt((comps_train ** 2).sum(axis=1))
    test_df['st_emb_l2'] = np.sqrt((comps_test ** 2).sum(axis=1))
    return train_df, test_df


def add_transformer_oof_text_predictions(train_df, test_df, y_values, weights=None):
    if not RUN_TRANSFORMER_TEXT_OOF or 'mentor_feedback_text' not in train_df.columns:
        return train_df, test_df
    try:
        import torch
        from torch.utils.data import DataLoader, Dataset
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    except Exception as e:
        print('Transformers import edilemedi, OOF text modeli atlandı:', repr(e))
        return train_df, test_df

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device != 'cuda' and not ALLOW_CPU_TRANSFORMER_OOF:
        print('GPU yok; transformer OOF fine-tune atlandı. RUN_SENTENCE_EMBEDDINGS + keyword featureları devam ediyor.')
        return train_df, test_df

    class EncodedTextDataset(Dataset):
        def __init__(self, enc, labels=None, sample_weight=None):
            self.enc = enc
            self.labels = labels
            self.sample_weight = sample_weight
        def __len__(self):
            return len(self.enc['input_ids'])
        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
                sw = 1.0 if self.sample_weight is None else float(self.sample_weight[idx])
                item['sample_weight'] = torch.tensor(sw, dtype=torch.float32)
            return item

    def predict_batches(model, loader):
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch).logits.squeeze(-1)
                preds.append(out.detach().cpu().numpy())
        return np.concatenate(preds)

    texts_train = train_df['mentor_feedback_text'].fillna('').astype(str).tolist()
    texts_test = test_df['mentor_feedback_text'].fillna('').astype(str).tolist()
    y_scaled = np.asarray(y_values, dtype=np.float32) / 100.0
    if weights is None:
        sw_all = np.ones(len(y_scaled), dtype=np.float32)
    else:
        sw_all = np.asarray(weights, dtype=np.float32)
        sw_all = sw_all / np.nanmean(sw_all)
    kf_text = KFold(n_splits=TRANSFORMER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE + 19)
    use_amp = device == 'cuda'

    for model_id in TRANSFORMER_MODEL_IDS:
        slug = cache_slug(model_id)
        col = f'text_oof_{slug}'
        cache_path = CACHE_DIR / f'{slug}_oof_{TRANSFORMER_N_SPLITS}fold_{TRANSFORMER_EPOCHS}ep_{TRANSFORMER_MAX_LENGTH}.npz'
        if cache_path.exists():
            data = np.load(cache_path)
            train_df[col] = data['oof']
            test_df[col] = data['test']
            print('Transformer OOF cache kullanıldı:', cache_path)
            continue

        print(f'Transformer OOF başlıyor: {model_id} | device={device}')
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        test_enc = tokenizer(texts_test, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
        test_loader = DataLoader(EncodedTextDataset(test_enc), batch_size=TRANSFORMER_BATCH_SIZE * 2, shuffle=False)
        oof = np.zeros(len(train_df), dtype=np.float32)
        test_pred = np.zeros(len(test_df), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(kf_text.split(texts_train), 1):
            tr_text = [texts_train[i] for i in tr_idx]
            va_text = [texts_train[i] for i in va_idx]
            tr_enc = tokenizer(tr_text, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
            va_enc = tokenizer(va_text, padding=True, truncation=True, max_length=TRANSFORMER_MAX_LENGTH, return_tensors='pt')
            tr_ds = EncodedTextDataset(tr_enc, y_scaled[tr_idx], sw_all[tr_idx])
            va_ds = EncodedTextDataset(va_enc)
            tr_loader = DataLoader(tr_ds, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True)
            va_loader = DataLoader(va_ds, batch_size=TRANSFORMER_BATCH_SIZE * 2, shuffle=False)

            model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=1, problem_type='regression')
            model.to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LR, weight_decay=TRANSFORMER_WEIGHT_DECAY)
            total_steps = max(1, TRANSFORMER_EPOCHS * len(tr_loader))
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps)
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

            for epoch in range(TRANSFORMER_EPOCHS):
                model.train()
                losses = []
                for batch in tr_loader:
                    labels = batch.pop('labels').to(device)
                    sample_weight = batch.pop('sample_weight').to(device)
                    batch = {k: v.to(device) for k, v in batch.items()}
                    optimizer.zero_grad(set_to_none=True)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        pred = model(**batch).logits.squeeze(-1)
                        loss = ((pred - labels) ** 2 * sample_weight).mean()
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    losses.append(float(loss.detach().cpu()))
                print(f'{slug} fold={fold} epoch={epoch+1}/{TRANSFORMER_EPOCHS} loss={np.mean(losses):.5f}')

            oof[va_idx] = np.clip(predict_batches(model, va_loader) * 100.0, 0, 100)
            test_pred += np.clip(predict_batches(model, test_loader) * 100.0, 0, 100) / TRANSFORMER_N_SPLITS
            del model, tr_enc, va_enc, tr_ds, va_ds, tr_loader, va_loader
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

        train_df[col] = oof
        test_df[col] = test_pred
        np.savez_compressed(cache_path, oof=oof, test=test_pred)
        print(f'{col}: MSE={np.mean((oof - y_values) ** 2):.4f} corr={np.corrcoef(oof, y_values)[0,1]:.4f}')
    return train_df, test_df


def weighted_group_mean(frame, key, y_values, weights=None):
    tmp = pd.DataFrame({key: frame[key].astype(str).values, '_y': y_values})
    if weights is None:
        return tmp.groupby(key)['_y'].mean()
    tmp['_w'] = weights
    tmp['_yw'] = tmp['_y'] * tmp['_w']
    sums = tmp.groupby(key)[['_yw', '_w']].sum()
    return sums['_yw'] / sums['_w']


def add_oof_target_encoding(train_df, test_df, cols, y_values, weights=None, n_splits=5):
    global_mean = np.average(y_values, weights=weights) if weights is not None else np.mean(y_values)
    kf_te = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE + 7)
    for c in cols:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        new_col = f'{c}_te'
        oof = np.full(len(train_df), global_mean, dtype=float)
        for tr_idx, va_idx in kf_te.split(train_df):
            means = weighted_group_mean(train_df.iloc[tr_idx], c, y_values[tr_idx], None if weights is None else weights[tr_idx])
            oof[va_idx] = train_df.iloc[va_idx][c].astype(str).map(means).fillna(global_mean).values
        full_means = weighted_group_mean(train_df, c, y_values, weights)
        train_df[new_col] = oof
        test_df[new_col] = test_df[c].astype(str).map(full_means).fillna(global_mean).values
    return train_df, test_df

before_cols = set(train.columns)
train = add_feature_rich(train)
test = add_feature_rich(test)
cat_for_unsup = [c for c in BASE_CAT + ['age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier'] if c in train.columns]
train, test = add_frequency_encoding(train, test, cat_for_unsup)
train, test = add_group_rank_features(train, test)
train, test = add_sentence_embedding_features(train, test)

# Test yıl dağılımını CV pusulasına yansıt: train yıllarını test dağılımına göre ağırlıklandır.
if 'application_year' in train.columns and 'application_year' in test.columns:
    yr_ratio = (test['application_year'].value_counts(normalize=True) /
                train['application_year'].value_counts(normalize=True))
    w = train['application_year'].map(yr_ratio).fillna(1.0).astype(float).values
else:
    yr_ratio = pd.Series(dtype=float)
    w = np.ones(len(train), dtype=float)
print('Yıl -> ağırlık:'); print(yr_ratio.sort_index().round(3))

train, test = add_transformer_oof_text_predictions(train, test, y, weights=w)

te_cols = [c for c in BASE_CAT + ['age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier'] if c in train.columns]
train, test = add_oof_target_encoding(train, test, te_cols, y, weights=w, n_splits=5)

# Drift etkileşimleri: tekil güçlü sinyallerin zaman içinde değişen etkisini modele açık ver.
def add_drift(df):
    df = df.copy()
    if 'application_year' not in df.columns:
        return df
    yr = pd.to_numeric(df['application_year'], errors='coerce') - 2019
    base_cols = ['project_quality_score', 'tech_mean', 'technical_interview_score', 'bert_pred',
                 'role_skill_fit', 'portfolio_score', 'experience_mean', 'mentor_pos_kw_count',
                 'mentor_neg_kw_count', 'mentor_sentiment_balance', 'career_prep_mean',
                 'text_oof_dbmdz_bert_base_turkish_cased',
                 'text_oof_dbmdz_electra_base_turkish_cased_discriminator']
    for c in base_cols:
        if c in df.columns:
            df[f'{c}_x_year'] = pd.to_numeric(df[c], errors='coerce') * yr
    return df

train = add_drift(train)
test = add_drift(test)

if 'bert_pred' in train.columns and 'bert_pred' in test.columns:
    bp = pd.to_numeric(train['bert_pred'], errors='coerce')
    print(f'bert_pred: corr={bp.corr(train[TARGET]):.3f} | MSE={((bp-train[TARGET])**2).mean():.1f}'
          f' | train mean={bp.mean():.1f} test mean={pd.to_numeric(test["bert_pred"], errors="coerce").mean():.1f}')
else:
    print('bert_pred yok; mentor metni transformer OOF + sentence embedding + keyword featurelarıyla temsil ediliyor.')

new_cols = sorted(set(train.columns) - before_cols)
print(f'Yeni feature sayısı: {len(new_cols)}')
print('Örnek yeni featurelar:', new_cols[:40])


def wmse(y_true, y_pred):
    return float(np.average((np.asarray(y_true) - np.asarray(y_pred))**2, weights=w))


## 2) CatBoost + LightGBM (ağırlıklı 5-fold)
CatBoost kategorikleri native işler; LightGBM tarafında aynı kategoriler pandas `category` olarak hizalanır. Bu hücre ayrıca LightGBM fold importance ile feature ailelerini tarar: transformer/text, role fit, target encoding, drift, group-rank ve domain interaction hangi ağırlıkta görünüyor raporlanır.


In [ ]:
# Hücre 3 — CatBoost ve LightGBM
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import lightgbm as lgb

CAT = [c for c in BASE_CAT + ['age_bucket', 'department__target_role', 'tier__target_role', 'age_bucket__target_role', 'department__tier'] if c in train.columns]
DROP_FOR_MODEL = {TARGET, ID_COL, 'mentor_feedback_text'}
feats = [c for c in train.columns if c not in DROP_FOR_MODEL]
obj_not_cat = [c for c in feats if str(train[c].dtype) in ('object', 'string') and c not in CAT]
if obj_not_cat:
    print('Model dışı bırakılan işlenmemiş object kolonları:', obj_not_cat)
    feats = [c for c in feats if c not in obj_not_cat]

for c in CAT:
    train[c] = train[c].fillna('__MISSING__').astype(str)
    test[c] = test[c].fillna('__MISSING__').astype(str)

X, Xte = train[feats], test[feats]
cat_idx = [feats.index(c) for c in CAT if c in feats]
kf = KFold(5, shuffle=True, random_state=2025)
print(f'n features: {len(feats)} | categoricals: {len(cat_idx)}')

DRIFT = [c for c in feats if c.endswith('_x_year')]


def run_catboost(feature_list):
    Xf, Xtef = train[feature_list], test[feature_list]
    ci = [feature_list.index(c) for c in CAT if c in feature_list]
    oof = np.zeros(len(Xf)); pred = np.zeros(len(Xtef))
    for fold, (ti, vi) in enumerate(kf.split(Xf), 1):
        m = CatBoostRegressor(
            iterations=4000, learning_rate=0.035, depth=6, loss_function='RMSE',
            l2_leaf_reg=4.0, random_strength=0.8,
            random_seed=RANDOM_STATE + fold, early_stopping_rounds=180, verbose=0,
            cat_features=ci, allow_writing_files=False
        )
        m.fit(Xf.iloc[ti], y[ti], sample_weight=w[ti], eval_set=(Xf.iloc[vi], y[vi]))
        oof[vi] = m.predict(Xf.iloc[vi]); pred += m.predict(Xtef) / 5
    return np.clip(oof, 0, 100), np.clip(pred, 0, 100)

# --- Drift ablasyonu: CatBoost'u iki feature setiyle koş, wmse'ye göre seç ---
if DRIFT:
    feats_nodrift = [c for c in feats if c not in DRIFT]
    oof_nd, pred_nd = run_catboost(feats_nodrift)
    oof_wd, pred_wd = run_catboost(feats)
    print(f'CatBoost DRIFTSIZ : wmse = {wmse(y, oof_nd):.4f} | düz MSE = {mean_squared_error(y, oof_nd):.4f}')
    print(f'CatBoost DRIFTLI  : wmse = {wmse(y, oof_wd):.4f} | düz MSE = {mean_squared_error(y, oof_wd):.4f}')
    if wmse(y, oof_wd) < wmse(y, oof_nd):
        print('>>> Drift etkileşimleri KALIYOR')
        oof_cb, pred_cb = oof_wd, pred_wd
    else:
        print('>>> Drift etkileşimleri kazanç vermedi, ATILIYOR (tüm modellerde)')
        oof_cb, pred_cb = oof_nd, pred_nd
        feats = feats_nodrift
        X, Xte = train[feats], test[feats]
else:
    oof_cb, pred_cb = run_catboost(feats)
    print(f'CatBoost : wmse = {wmse(y, oof_cb):.4f} | düz MSE = {mean_squared_error(y, oof_cb):.4f}')

# --- LightGBM ---
Xl = X.copy(); Xlte = Xte.copy()
for c in [c for c in CAT if c in Xl.columns]:
    categories = pd.Index(pd.concat([Xl[c], Xlte[c]], axis=0).astype(str).unique())
    dtype = pd.CategoricalDtype(categories=categories)
    Xl[c] = Xl[c].astype(str).astype(dtype)
    Xlte[c] = Xlte[c].astype(str).astype(dtype)

oof_lgb = np.zeros(len(Xl)); pred_lgb = np.zeros(len(Xlte)); lgb_importances = []
for fold, (ti, vi) in enumerate(kf.split(Xl), 1):
    m = lgb.LGBMRegressor(
        n_estimators=6000, learning_rate=0.022, num_leaves=63,
        min_child_samples=30, reg_alpha=0.05, reg_lambda=1.2,
        colsample_bytree=0.82, subsample=0.82, subsample_freq=1,
        random_state=RANDOM_STATE + fold, verbose=-1
    )
    m.fit(Xl.iloc[ti], y[ti], sample_weight=w[ti], eval_set=[(Xl.iloc[vi], y[vi])],
          categorical_feature=[c for c in CAT if c in Xl.columns],
          callbacks=[lgb.early_stopping(220, verbose=False)])
    oof_lgb[vi] = m.predict(Xl.iloc[vi]); pred_lgb += m.predict(Xlte) / 5
    lgb_importances.append(pd.Series(m.feature_importances_, index=Xl.columns))

oof_lgb = np.clip(oof_lgb, 0, 100); pred_lgb = np.clip(pred_lgb, 0, 100)
print(f'LightGBM : wmse = {wmse(y, oof_lgb):.4f} | düz MSE = {mean_squared_error(y, oof_lgb):.4f}')

imp = pd.concat(lgb_importances, axis=1).mean(axis=1).sort_values(ascending=False)
print('\nTop 40 LightGBM feature importance:')
print(imp.head(40).round(1).to_string())


def feature_family(col):
    if col.startswith('text_oof_') or col.startswith('st_emb_'):
        return 'transformer_text'
    if col.startswith('mentor_'):
        return 'mentor_keywords'
    if col.endswith('_te'):
        return 'target_encoding'
    if col.endswith('_x_year'):
        return 'year_drift'
    if '_rank_in_' in col or '_z_in_' in col:
        return 'group_rank_z'
    if col in CAT or col.endswith('_freq'):
        return 'categorical_freq'
    if 'role_' in col:
        return 'role_fit'
    if '_x_' in col or 'gap' in col or 'rate' in col or 'ratio' in col or 'product' in col:
        return 'interaction_ratio'
    if any(col.startswith(p) for p in ['tech_', 'core_dev_', 'data_ai_', 'web_dev_', 'infra_', 'soft_', 'interview_', 'portfolio_', 'experience_', 'career_prep_', 'academic_']):
        return 'domain_composite'
    return 'raw_or_other'

fam = imp.groupby(imp.index.map(feature_family)).sum().sort_values(ascending=False)
print('\nFeature family importance:')
print(fam.round(1).to_string())


# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    Xx = X.copy(); Xxte = Xte.copy()
    for c in [c for c in CAT if c in Xx.columns]:
        categories = pd.Index(pd.concat([Xx[c], Xxte[c]], axis=0).astype(str).unique())
        dtype = pd.CategoricalDtype(categories=categories)
        Xx[c] = Xx[c].astype(str).astype(dtype)
        Xxte[c] = Xxte[c].astype(str).astype(dtype)

    oof_xgb = np.zeros(len(Xx)); pred_xgb = np.zeros(len(Xxte)); xgb_importances = []
    for fold, (ti, vi) in enumerate(kf.split(Xx), 1):
        m = XGBRegressor(
            n_estimators=3500, learning_rate=0.025, max_depth=5,
            min_child_weight=8, subsample=0.86, colsample_bytree=0.82,
            reg_alpha=0.02, reg_lambda=1.5, objective='reg:squarederror',
            tree_method='hist', enable_categorical=True, eval_metric='rmse',
            early_stopping_rounds=180, random_state=RANDOM_STATE + fold,
            n_jobs=-1
        )
        m.fit(Xx.iloc[ti], y[ti], sample_weight=w[ti], eval_set=[(Xx.iloc[vi], y[vi])], verbose=False)
        oof_xgb[vi] = m.predict(Xx.iloc[vi]); pred_xgb += m.predict(Xxte) / 5
        try:
            xgb_importances.append(pd.Series(m.feature_importances_, index=Xx.columns))
        except Exception:
            pass
    oof_xgb = np.clip(oof_xgb, 0, 100); pred_xgb = np.clip(pred_xgb, 0, 100)
    print(f'XGBoost  : wmse = {wmse(y, oof_xgb):.4f} | düz MSE = {mean_squared_error(y, oof_xgb):.4f}')
    if xgb_importances:
        imp_xgb = pd.concat(xgb_importances, axis=1).mean(axis=1).sort_values(ascending=False)
        print('\nTop 25 XGBoost feature importance:')
        print(imp_xgb.head(25).round(4).to_string())
except Exception as e:
    print('XGBoost atlandı:', repr(e))


## 3) AutoGluon (ağırlıklı, DyStack kapalı)
AutoGluon aynı feature setiyle çalışır. Text transformer tahminleri ve embedding bileşenleri artık normal sayısal kolonlar olduğu için AutoGluon bunları CatBoost/LightGBM/NN ensemble içinde kullanabilir.


In [ ]:
# Hücre 4 — AutoGluon
from autogluon.tabular import TabularPredictor
import torch

train_ag = train[feats + [TARGET]].copy()
test_ag = test[feats].copy()
train_ag['sample_weight'] = w

ag_num_gpus = 1 if torch.cuda.is_available() else 0
ag_path = 'AutogluonModels_local' if LOCAL_RUN else 'AutogluonModels'

predictor = TabularPredictor(
    label=TARGET, eval_metric='root_mean_squared_error',
    sample_weight='sample_weight', weight_evaluation=True,
    path=ag_path
).fit(
    train_data=train_ag, time_limit=TIME_LIMIT, presets='best_quality',
    num_bag_folds=5, dynamic_stacking=False, num_stack_levels=0,
    num_gpus=ag_num_gpus,
)

lb = predictor.leaderboard(silent=True)
print(lb[['model','score_val']].head(10))

pred_ag = np.clip(predictor.predict(test_ag).values, 0, 100)
try:
    oof_ag = np.clip(predictor.predict_oof().values, 0, 100)
except AttributeError:
    oof_ag = np.clip(predictor.get_oof_pred().values, 0, 100)
print(f'AutoGluon: wmse = {wmse(y, oof_ag):.4f} | düz MSE = {mean_squared_error(y, oof_ag):.4f}')


## 4) Model seçimi + ağırlıklı Ridge stack → submission
Stack yalnızca OOF tahminleriyle eğitilir. Son seçim ağırlıklı CV pusulasına göre yapılır; resmi metrik MSE olduğu için düz MSE de raporlanır.


In [ ]:
# Hücre 5 — advanced stacking + calibration sweep + submission
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize

base_models = [
    ('autogluon', oof_ag, pred_ag),
    ('catboost', oof_cb, pred_cb),
    ('lightgbm', oof_lgb, pred_lgb),
]
if 'oof_xgb' in globals() and 'pred_xgb' in globals():
    base_models.append(('xgboost', oof_xgb, pred_xgb))
print('Stack base modeller:', [m[0] for m in base_models])

S_oof  = np.column_stack([m[1] for m in base_models])
S_test = np.column_stack([m[2] for m in base_models])
base_names = [m[0] for m in base_models]

meta_oof = np.zeros(len(S_oof)); meta_pred = np.zeros(len(S_test))
meta_weights = []
for ti, vi in kf.split(S_oof):
    r = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(S_oof[ti], y[ti], sample_weight=w[ti])
    meta_oof[vi] = r.predict(S_oof[vi]); meta_pred += r.predict(S_test) / 5
    meta_weights.append(r.coef_)
meta_oof = np.clip(meta_oof, 0, 100); meta_pred = np.clip(meta_pred, 0, 100)
print('Ortalama stack katsayıları', base_names, ':', np.mean(meta_weights, axis=0).round(3))

# Global OOF-weight optimizer. It is a small 3-4 weight validation blend; use with CV/LB sanity.
def optimize_blend(metric='wmse'):
    n = S_oof.shape[1]
    x0 = np.ones(n) / n
    bounds = [(0.0, 1.0)] * n
    cons = [{'type': 'eq', 'fun': lambda z: np.sum(z) - 1.0}]
    def obj(z):
        pred = S_oof @ z
        err = (y - pred) ** 2
        return float(np.average(err, weights=w)) if metric == 'wmse' else float(np.mean(err))
    res = minimize(obj, x0, method='SLSQP', bounds=bounds, constraints=cons, options={'maxiter': 1000, 'ftol': 1e-12})
    weights_opt = res.x if res.success else x0
    return np.clip(S_oof @ weights_opt, 0, 100), np.clip(S_test @ weights_opt, 0, 100), weights_opt

opt_w_oof, opt_w_pred, opt_w = optimize_blend('wmse')
opt_m_oof, opt_m_pred, opt_m = optimize_blend('mse')
print('Optimize blend wmse weights:', dict(zip(base_names, np.round(opt_w, 4))))
print('Optimize blend mse  weights:', dict(zip(base_names, np.round(opt_m, 4))))

cands = {
    'stack_ridge': (wmse(y, meta_oof), mean_squared_error(y, meta_oof), meta_pred, meta_oof),
    'blend_opt_wmse': (wmse(y, opt_w_oof), mean_squared_error(y, opt_w_oof), opt_w_pred, opt_w_oof),
    'blend_opt_mse': (wmse(y, opt_m_oof), mean_squared_error(y, opt_m_oof), opt_m_pred, opt_m_oof),
}
for name, oof_pred, test_pred in base_models:
    cands[name] = (wmse(y, oof_pred), mean_squared_error(y, oof_pred), test_pred, oof_pred)


def fold_safe_poly_calibrate(base_oof, base_pred, degree=2, alpha=1.0):
    cal_oof = np.zeros_like(base_oof, dtype=float)
    for ti, vi in kf.split(base_oof):
        Ptr = np.column_stack([base_oof[ti] ** d for d in range(1, degree + 1)])
        Pva = np.column_stack([base_oof[vi] ** d for d in range(1, degree + 1)])
        model = Ridge(alpha=alpha).fit(Ptr, y[ti], sample_weight=w[ti])
        cal_oof[vi] = model.predict(Pva)
    Pfull = np.column_stack([base_oof ** d for d in range(1, degree + 1)])
    Ptest = np.column_stack([base_pred ** d for d in range(1, degree + 1)])
    model = Ridge(alpha=alpha).fit(Pfull, y, sample_weight=w)
    cal_pred = model.predict(Ptest)
    return np.clip(cal_oof, 0, 100), np.clip(cal_pred, 0, 100)


def fold_safe_isotonic(base_oof, base_pred):
    cal_oof = np.zeros_like(base_oof, dtype=float)
    for ti, vi in kf.split(base_oof):
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=100)
        iso.fit(base_oof[ti], y[ti], sample_weight=w[ti])
        cal_oof[vi] = iso.predict(base_oof[vi])
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=100)
    iso.fit(base_oof, y, sample_weight=w)
    cal_pred = iso.predict(base_pred)
    return np.clip(cal_oof, 0, 100), np.clip(cal_pred, 0, 100)


def fold_safe_predbin_residual(base_oof, base_pred, n_bins=10, shrink=120.0):
    cal_oof = np.zeros_like(base_oof, dtype=float)
    for ti, vi in kf.split(base_oof):
        q = np.unique(np.quantile(base_oof[ti], np.linspace(0, 1, n_bins + 1)))
        if len(q) < 3:
            cal_oof[vi] = base_oof[vi]
            continue
        bins_tr = pd.cut(base_oof[ti], bins=q, include_lowest=True, labels=False)
        bins_va = pd.cut(base_oof[vi], bins=q, include_lowest=True, labels=False)
        resid = y[ti] - base_oof[ti]
        global_offset = float(np.average(resid, weights=w[ti]))
        tmp = pd.DataFrame({'bin': bins_tr, 'resid': resid, 'w': w[ti]}).dropna()
        tmp['_rw'] = tmp['resid'] * tmp['w']
        stats = tmp.groupby('bin').agg(sum_rw=('_rw', 'sum'), sum_w=('w', 'sum'))
        offsets = (stats['sum_rw'] + shrink * global_offset) / (stats['sum_w'] + shrink)
        cal_oof[vi] = base_oof[vi] + pd.Series(bins_va).map(offsets).fillna(global_offset).values
    q = np.unique(np.quantile(base_oof, np.linspace(0, 1, n_bins + 1)))
    if len(q) < 3:
        return np.clip(cal_oof, 0, 100), np.clip(base_pred, 0, 100)
    bins_tr = pd.cut(base_oof, bins=q, include_lowest=True, labels=False)
    bins_te = pd.cut(base_pred, bins=q, include_lowest=True, labels=False)
    resid = y - base_oof
    global_offset = float(np.average(resid, weights=w))
    tmp = pd.DataFrame({'bin': bins_tr, 'resid': resid, 'w': w}).dropna()
    tmp['_rw'] = tmp['resid'] * tmp['w']
    stats = tmp.groupby('bin').agg(sum_rw=('_rw', 'sum'), sum_w=('w', 'sum'))
    offsets = (stats['sum_rw'] + shrink * global_offset) / (stats['sum_w'] + shrink)
    cal_pred = base_pred + pd.Series(bins_te).map(offsets).fillna(global_offset).values
    return np.clip(cal_oof, 0, 100), np.clip(cal_pred, 0, 100)


def _group_key(df, groups):
    return df[groups].astype(str).agg('||'.join, axis=1)


def _fit_residual_offsets(frame, residual, groups, sample_weight, shrink=100.0):
    key = _group_key(frame, groups)
    tmp = pd.DataFrame({'key': key.values, 'resid': residual, 'w': sample_weight})
    tmp['_rw'] = tmp['resid'] * tmp['w']
    global_offset = float(np.average(tmp['resid'], weights=tmp['w']))
    stats = tmp.groupby('key').agg(sum_rw=('_rw', 'sum'), sum_w=('w', 'sum'))
    offsets = (stats['sum_rw'] + shrink * global_offset) / (stats['sum_w'] + shrink)
    return offsets, global_offset


def _apply_residual_offsets(frame, groups, offsets, default_offset):
    key = _group_key(frame, groups)
    return key.map(offsets).fillna(default_offset).astype(float).values


def residual_calibrate(base_oof, base_pred, groups, shrink=100.0):
    cal_oof = np.zeros_like(base_oof, dtype=float)
    for ti, vi in kf.split(train):
        offsets, default = _fit_residual_offsets(train.iloc[ti], y[ti] - base_oof[ti], groups, w[ti], shrink=shrink)
        cal_oof[vi] = base_oof[vi] + _apply_residual_offsets(train.iloc[vi], groups, offsets, default)
    offsets, default = _fit_residual_offsets(train, y - base_oof, groups, w, shrink=shrink)
    cal_pred = base_pred + _apply_residual_offsets(test, groups, offsets, default)
    return np.clip(cal_oof, 0, 100), np.clip(cal_pred, 0, 100)

# Calibrate the strongest stack/blends in several ways.
primary_for_cal = {
    'stack_ridge': (meta_oof, meta_pred),
    'blend_opt_wmse': (opt_w_oof, opt_w_pred),
}
for base_name, (base_oof, base_pred) in primary_for_cal.items():
    for degree, alpha in [(2, 1.0), (3, 10.0)]:
        cal_oof, cal_pred = fold_safe_poly_calibrate(base_oof, base_pred, degree=degree, alpha=alpha)
        cands[f'{base_name}_poly{degree}'] = (wmse(y, cal_oof), mean_squared_error(y, cal_oof), cal_pred, cal_oof)
    cal_oof, cal_pred = fold_safe_isotonic(base_oof, base_pred)
    cands[f'{base_name}_isotonic'] = (wmse(y, cal_oof), mean_squared_error(y, cal_oof), cal_pred, cal_oof)
    for bins in [8, 12, 16]:
        for shrink in [80.0, 180.0, 400.0]:
            cal_oof, cal_pred = fold_safe_predbin_residual(base_oof, base_pred, n_bins=bins, shrink=shrink)
            cands[f'{base_name}_predbin{bins}_s{int(shrink)}'] = (wmse(y, cal_oof), mean_squared_error(y, cal_oof), cal_pred, cal_oof)

calibration_groups = {
    'year': ['application_year'],
    'role': ['target_role'],
    'tier': ['university_tier'],
    'year_role': ['application_year', 'target_role'],
    'tier_role': ['university_tier', 'target_role'],
    'dept_role': ['department', 'target_role'],
    'year_tier_role': ['application_year', 'university_tier', 'target_role'],
    'year_dept_role': ['application_year', 'department', 'target_role'],
}
for base_name, (base_oof, base_pred) in primary_for_cal.items():
    for group_name, groups in calibration_groups.items():
        if not all(g in train.columns and g in test.columns for g in groups):
            continue
        for shrink in [80.0, 180.0, 250.0, 500.0, 900.0]:
            cal_oof, cal_pred = residual_calibrate(base_oof, base_pred, groups, shrink=shrink)
            name = f'{base_name}_cal_{group_name}_s{int(shrink)}'
            cands[name] = (wmse(y, cal_oof), mean_squared_error(y, cal_oof), cal_pred, cal_oof)

print('\nModel adayları:')
for k in sorted(cands, key=lambda key: cands[key][0]):
    s_w, s_mse, _, _ = cands[k]
    print(f'{k:42s} wmse = {s_w:.4f} | MSE = {s_mse:.4f}')

best_name = min(cands, key=lambda k: cands[k][0])
best_wmse, best_mse, best_pred, best_oof = cands[best_name]
print('Secilen:', best_name, '| Beklenen LB ~', round(best_wmse, 1), '(±2)')


def _weighted_mean(values, weights):
    return float(np.average(np.asarray(values), weights=np.asarray(weights)))


def residual_segment_report(pred_oof, report_cols=('application_year', 'target_role', 'university_tier')):
    resid = y - pred_oof
    print('\nResidual diagnostics — seçilen aday:', best_name)
    print(f'global bias={_weighted_mean(resid, w):+.4f} | weighted abs err={_weighted_mean(np.abs(resid), w):.4f}')
    for col in report_cols:
        if col not in train.columns:
            continue
        rows = []
        tmp = pd.DataFrame({col: train[col].astype(str), 'y': y, 'pred': pred_oof, 'resid': resid, 'abs_resid': np.abs(resid), 'w': w})
        for val, g in tmp.groupby(col):
            rows.append({
                col: val,
                'n': len(g),
                'mean_y': np.average(g['y'], weights=g['w']),
                'mean_pred': np.average(g['pred'], weights=g['w']),
                'bias_y_minus_pred': np.average(g['resid'], weights=g['w']),
                'w_abs_err': np.average(g['abs_resid'], weights=g['w']),
            })
        rep = pd.DataFrame(rows).sort_values('w_abs_err', ascending=False)
        print(f'\nTop residual segments by {col}:')
        print(rep.head(12).round(3).to_string(index=False))

residual_segment_report(best_oof)

sub = pd.DataFrame({ID_COL: test_ids, TARGET: best_pred})
out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
sub.to_csv(out / 'submission.csv', index=False)
print('\nSubmission sample:')
print(sub.head(3))
print(f'Toplam sure: {(time.time()-t0)/60:.1f} dk')


## Araştırma notları ve sıradaki deneyler
- Local EDA'da `project_quality_score` hedefle en yüksek tekil korelasyonu verdi; `technical_interview_score`, teknik skorlar, portfolyo/GitHub ve gerçek proje sinyalleri onu takip ediyor.
- Mentor metnindeki `mükemmel`, `olağanüstü`, `başarı`, `yüksek` gibi ifadeler pozitif; `eksik`, `geliştirmesi`, `ihtiyaç`, `gerekiyor` gibi ifadeler negatif lift taşıyor.
- Metin için üç katman var: keyword/topic sinyalleri, sentence-transformer embedding, fold-dışı BERTurk/ELECTRA regresyon tahmini. En kritik kalite sigortası: text OOF train tahminleri asla aynı satırın hedefiyle eğitilmiş modelden gelmemeli.
- Test seti 2024-2026 yıllarına train'e göre belirgin daha fazla yığılıyor; bu yüzden `wmse` yıl ağırlığı ve yıl etkileşimleri tutuldu.
- Yarışma modu deney sırası: önce BERTurk tek başına, sonra ELECTRA eklenmiş hali; `SENTENCE_EMBEDDING_COMPONENTS` için 32/64/128 ablation; `TRANSFORMER_EPOCHS` için 2/3 ablation; son olarak 2 seed CatBoost/LightGBM bagging.
